# Gold : `fact_order_items`

**Entrées :** `workspace.silver.silver_order_items`, `workspace.gold.fact_orders`, `workspace.gold.dim_products`
**Sortie :** `workspace.gold.fact_order_items` (**une ligne par article**, clé : `order_id` + `order_item_id`)

## Pourquoi cette table
`fact_orders` a une ligne par commande, mais une commande peut contenir plusieurs produits de catégories différentes. Pour calculer le chiffre d'affaires **par catégorie**, il faut le grain « article ».

## Ce qui change de Silver vers Gold
| Silver | Gold |
|---|---|
| `silver_order_items` : prix, port, produit, vendeur | Ajout du statut, de `is_sale`, de la date et du mois de la commande |
| Aucune catégorie | `category` (anglais) tirée de `dim_products` |
| Prix et port séparés | `item_total` = prix + port |

## Règles
- **Agréger avant de joindre** : `fact_orders` (une ligne par commande) et `dim_products` (une ligne par produit) sont uniques sur leur clé. Les `LEFT JOIN` ne peuvent donc pas dupliquer d'article.
- Le chiffre d'affaires est le **prix** des articles (`price`), sans les frais de port.
- **Définition provisoire d'une vente** : commande ni `canceled` ni `unavailable` (comme dans les KPI).

In [0]:
CREATE OR REPLACE TABLE workspace.gold.fact_order_items AS
SELECT
    i.order_id,
    i.order_item_id,
    i.product_id,
    i.seller_id,
    o.customer_unique_id,
    o.order_status,
    o.is_sale,
    o.order_date,
    o.order_month,
    COALESCE(p.category, 'unknown_category') AS category,
    i.price,
    i.freight_value,
    i.price + i.freight_value                 AS item_total
FROM workspace.silver.silver_order_items i
LEFT JOIN workspace.gold.fact_orders o ON i.order_id = o.order_id
LEFT JOIN workspace.gold.dim_products p ON i.product_id = p.product_id

num_affected_rows,num_inserted_rows


## Vérification
| Mesure | Attendu |
|---|---:|
| `lignes`, `cles_distinctes` | 112 650 |
| `sans_commande` | 0 |
| `sans_produit` | 0 |
| `ca_gold` = `ca_silver` | 13 591 643,70, `identiques` = true |
| `ca_ventes` | 13 494 400,74 (même chiffre que `kpi_monthly_revenue`) |

In [0]:
SELECT
    COUNT(*) AS lignes,
    COUNT(DISTINCT order_id, order_item_id) AS cles_distinctes,
    COUNT(*) FILTER (WHERE order_status IS NULL) AS sans_commande,
    COUNT(*) FILTER (WHERE category = 'unknown_category' AND product_id NOT IN (SELECT product_id FROM workspace.gold.dim_products)) AS sans_produit,
    COUNT(DISTINCT category) AS categories
FROM workspace.gold.fact_order_items

lignes,cles_distinctes,sans_commande,sans_produit,categories
112650,112650,0,0,74


In [0]:
SELECT
    (SELECT SUM(price) FROM workspace.gold.fact_order_items)                      AS ca_gold,
    (SELECT SUM(price) FROM workspace.silver.silver_order_items)                  AS ca_silver,
    (SELECT SUM(price) FROM workspace.gold.fact_order_items)
      = (SELECT SUM(price) FROM workspace.silver.silver_order_items)              AS identiques,
    (SELECT SUM(price) FROM workspace.gold.fact_order_items WHERE is_sale)        AS ca_ventes,
    (SELECT SUM(product_revenue) FROM workspace.gold.kpi_monthly_revenue)         AS ca_kpi_mensuel

ca_gold,ca_silver,identiques,ca_ventes,ca_kpi_mensuel
13591643.70,13591643.70,true,13494400.74,13494400.74
